In [65]:
import os
import re
import json
import requests
from bs4 import BeautifulSoup
from collections import OrderedDict

from debate_info import debate_info
from debate_dict import debate_dict

In [156]:
import pandas as pd

def get_raw():
    raw_info = []
    with open('scraped_info.txt', 'r') as info_file:
        lines = info_file.readlines()
        num_lines = len(lines)
        i = 0
        year_debate_count = {}
        while i < num_lines:
            file_line = lines[i].strip()
            # Check if line contains a filename of the expected format
            if file_line.count('-') >= 2 and not file_line.endswith("-info.txt"): 
                # Extract year, month and day from filename
                year, month, day, _ = file_line.split('-')[:4]
                id = year_debate_count.get(year, 0) + 1
                year_debate_count[year] = id
                sentence_count = int(lines[i+1].split(': ')[1])
                word_count = int(lines[i+2].split(': ')[1])
                raw_info.append([year, month, day, id, word_count, sentence_count])
            i += 3  # Skip to the next file

    raw_df = pd.DataFrame(raw_info, columns=['YYYY', 'MM', 'DD', 'id', 'word_count', 'sentence_count'])
    return raw_df

raw_info = get_raw()


In [158]:
sorted_df = raw_info.sort_values(['YYYY','MM', 'DD'], ascending=True)
sorted_df = sorted_df.reset_index(drop=True)

sorted_df

,YYYY,MM,DD,id,word_count,sentence_count
0,1960,09,26,3,12454,525
1,1960,10,07,1,12811,530
2,1960,10,13,2,12421,506
3,1960,10,21,4,12451,554
4,1976,09,23,3,16656,634
5,1976,10,06,1,15202,622
6,1976,10,22,2,16250,637
7,1980,09,21,1,11918,461
8,1980,10,28,2,17579,684
9,1984,10,07,2,18507,846


In [ ]:
import os

# Replace with your directory
directory = "all_debates_raw"

raw_files = []

# Walk through all subdirectories
for root, dirs, files in os.walk(directory):
    for file in files:
        if file.endswith("-raw.txt"):
            raw_files.append(os.path.join(root, file))

# Print all raw files
for file in raw_files:
    print(file)


In [68]:
s292020 = {
  "Debate": "General Election Presidential Debate",
  "Participants": ["Donald J. Trump (R)", "Joseph R. Biden (D)"],
  "Date": "September 29, 2020",
  "Location": "Health Education Campus (HEC) Samson Pavilion",
  "City": "Cleveland, OH",
  "Time": "9:00-10:30 p.m. Eastern",
  "Sponsor": "Commission on Presidential Debates",
  "Moderator": "Chris Wallace, FOX",
  "Topics": "The records of President Trump and former Vice President Biden, the Supreme Court, COVID-19, the economy, race and violence in our cities, and the integrity of the election",
  "Viewership": "73.1 million (Data provided by Nielsen Media Research)",
  "Format": "Six 15-minute segments, each dedicated to a topic announced in advance.",
}

o72020 = {
  "Debate": "Vice Presidential Debate",
  "Participants": ["Mike Pence (R)", "Kamala Harris (D)"],
  "Date": "October 7, 2020",
  "Location": "University of Utah",
  "City": "Salt Lake City, UT",
  "Time": "9:00-10:30 p.m. Eastern",
  "Sponsor": "Commission on Presidential Debates",
  "Moderator": "Susan Page, USA Today",
  "Topics": "All Topics",
  "Viewership": "57.9 million (Data provided by Nielsen Media Research)",
  "Format": "Nine 10-minute segments, each dedicated to a topic announced in advance.",
}

o222020 = {
  "Debate": "General Election Presidential Debate",
  "Participants": ["Donald J. Trump (R)", "Joseph R. Biden (D)"],
  "Date": "October 22, 2020",
  "Location": "Curb Event Center at Belmont University",
  "City": "Nashville, TN",
  "Time": "9:00-10:30 p.m. Eastern",
  "Sponsor": "Commission on Presidential Debates",
  "Moderator": "Kristen Welker, NBC",
  "Topics": "Fighting COVID-19, American Families, Race in America, Climate Change, National Security, and Leadership",
  "Viewership": "63 million (Data provided by Nielsen Media Research)",
  "Format": "Six 15-minute segments, each dedicated to a topic announced in advance.",
}

In [116]:
from collections import OrderedDict
import re
from datetime import datetime

# Original dictionaries
dicts = [s292020, o72020, o222020]

# Function to strip titles and middle initials
def strip_name(name):
    name = re.sub(r'\(.*\)', '', name)  # Remove party affiliation
    name = re.sub(r'\b[A-Z]\.\s', '', name)  # Remove middle initials
    name = re.sub(r'(Mr\.|President|Governor)\s', '', name)  # Remove titles
    return name.strip()

# Function to create nested dictionary for candidates
def create_candidate_dict(name):
    party = re.search(r'\((.*?)\)', name).group(1)  # Extract party affiliation
    stripped_name = strip_name(name)
    symbol = stripped_name.split()[-1].upper()  # Last name in uppercase
    return { 'name': stripped_name, 'party': party, 'symbol': symbol }

# Function to create nested dictionary for moderator
def create_moderator_dict(moderator):
    name, affiliation = moderator.split(', ')
    stripped_name = strip_name(name)
    symbol = stripped_name.split()[-1].upper()  # Last name in uppercase
    return { 'name': name, 'affiliation': affiliation, 'symbol': symbol }

# Merged dictionary
debates_2020 = OrderedDict()

for d in dicts:
    # Create new primary key with formatted date
    date = d["Date"]
    formatted_date = datetime.strptime(date, "%B %d, %Y").strftime('%Y%m%d')  # 'YYYYMMDD'
    
    # Create new dictionary with lower case keys and updated values
    new_dict = { key.lower(): value for key, value in d.items() }
    new_dict['candidates'] = [create_candidate_dict(name) for name in new_dict.pop('participants')]
    new_dict['moderator'] = create_moderator_dict(new_dict['moderator'])
    
    # Split 'city' into 'city' and 'state'
    city, state = new_dict['city'].split(', ')
    new_dict['city'] = city
    new_dict['state'] = state
    
    # Split 'viewership' into 'viewership' and 'data_provider'
    viewership, data_provider = new_dict['viewership'].split(' (Data provided by ')
    new_dict['viewership'] = viewership.split()[0]  # Only keep the number
    new_dict['data_provider'] = data_provider.rstrip(')')  # Remove trailing parenthesis
    
    # Add to merged dictionary using formatted date as key
    debates_2020[formatted_date] = new_dict

# Order by descending date
debates_2020 = OrderedDict(sorted(debates_2020.items(), reverse=True))


In [117]:
debates_2020['20200929']

{'debate': 'General Election Presidential Debate',
 'date': 'September 29, 2020',
 'location': 'Health Education Campus (HEC) Samson Pavilion',
 'city': 'Cleveland',
 'time': '9:00-10:30 p.m. Eastern',
 'sponsor': 'Commission on Presidential Debates',
 'moderator': {'name': 'Chris Wallace',
  'affiliation': 'FOX',
  'symbol': 'WALLACE'},
 'topics': 'The records of President Trump and former Vice President Biden, the Supreme Court, COVID-19, the economy, race and violence in our cities, and the integrity of the election',
 'viewership': '73.1',
 'format': 'Six 15-minute segments, each dedicated to a topic announced in advance.',
 'candidates': [{'name': 'Donald Trump', 'party': 'R', 'symbol': 'TRUMP'},
  {'name': 'Joseph Biden', 'party': 'D', 'symbol': 'BIDEN'}],
 'state': 'OH',
 'data_provider': 'Nielsen Media Research'}

In [118]:
import pandas as pd

# Convert the OrderedDict to a list of dictionaries
debates_list = list(debates_2020.values())

# Flatten the list of dictionaries and convert to DataFrame
df = pd.json_normalize(debates_list)

# Set the index to the 'date' column
df.set_index('date', inplace=True)

# Display the DataFrame
df

,debate,location,city,time,sponsor,topics,viewership,format,candidates,state,data_provider,moderator.name,moderator.affiliation,moderator.symbol
date,,,,,,,,,,,,,,
"October 22, 2020",General Election Presidential Debate,Curb Event Center at Belmont University,Nashville,9:00-10:30 p.m. Eastern,Commission on Presidential Debates,"Fighting COVID-19, American Families, Race in ...",63,"Six 15-minute segments, each dedicated to a to...","[{'name': 'Donald Trump', 'party': 'R', 'symbo...",TN,Nielsen Media Research,Kristen Welker,NBC,WELKER
"October 7, 2020",Vice Presidential Debate,University of Utah,Salt Lake City,9:00-10:30 p.m. Eastern,Commission on Presidential Debates,All Topics,57.9,"Nine 10-minute segments, each dedicated to a t...","[{'name': 'Mike Pence', 'party': 'R', 'symbol'...",UT,Nielsen Media Research,Susan Page,USA Today,PAGE
"September 29, 2020",General Election Presidential Debate,Health Education Campus (HEC) Samson Pavilion,Cleveland,9:00-10:30 p.m. Eastern,Commission on Presidential Debates,The records of President Trump and former Vice...,73.1,"Six 15-minute segments, each dedicated to a to...","[{'name': 'Donald Trump', 'party': 'R', 'symbo...",OH,Nielsen Media Research,Chris Wallace,FOX,WALLACE
